In [ ]:
# Imports
import sys
from pathlib import Path
import warnings
import os
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="X does not have valid feature names")

ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.tabsyn.models import TabSyn


dataset_path = ROOT / "raw_data" / "magic.csv"
output_path  = ROOT / "discretized_data" / "magic.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path))


input_csv     = str(output_path)
output_dir    = str(ROOT / "sample_data" / "magic")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "magic" / "tabsyn")


def ensure_tabsyn_npy(data_dir: str):
    for split in ["train", "test"]:
        y_csv = os.path.join(data_dir, f"y_{split}.csv")
        y_npy = os.path.join(data_dir, f"y_{split}.npy")

        x_csv = os.path.join(data_dir, f"x_{split}.csv")
        xcat_npy = os.path.join(data_dir, f"X_cat_{split}.npy")

        if os.path.exists(y_csv) and not os.path.exists(y_npy):
            y = pd.read_csv(y_csv).iloc[:, 0].astype(int).to_numpy()
            np.save(y_npy, y, allow_pickle=True)

        if os.path.exists(x_csv) and not os.path.exists(xcat_npy):
            X = pd.read_csv(x_csv).astype(str).to_numpy()
            np.save(xcat_npy, X, allow_pickle=True)

# Run pipeline
pipeline = TrainTestSplitPipeline(
    model=lambda: TabSyn(
        d_token=32,
        decoder_epochs=100,
        diffusion_epochs=100,
        diffusion_steps=1000,
        decoder_batch_size=512,
        diffusion_batch_size=512,
        lr= 2e-3,
        weight_decay=1e-3,
        patience=80,
        seed=42,
        device="auto"
    )
)


try:
    result = pipeline.run(
        input_csv=input_csv,       
        output_dir=output_dir,
        synthetic_dir=synthetic_dir,
        real_test_dir=real_test_dir,
        size_category="medium",
    )
    print(result)

except FileNotFoundError:
    
    ensure_tabsyn_npy(output_dir)

    result = pipeline.run(
        input_csv=input_csv,
        output_dir=output_dir,
        synthetic_dir=synthetic_dir,
        real_test_dir=real_test_dir,
        size_category="medium",
    )
    print(result)
   

ROOT set to: C:\Users\Prabu\Downloads\Katabatic
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\magic.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\magic.csv
Loaded data with shape: (19020, 11)
Saved train/test full data
Train size: (15216, 11), Test size: (3804, 11)
Train label distribution:
 class
0    0.648396
1    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.648265
1    0.351735
Name: proportion, dtype: float64
Saved X/y split
Training shape: (15216, 10) (15216,)
Test shape: (3804, 10) (3804,)
